# AI Docstring & Unit Test Generator

A code tool that takes raw Python functions and automatically generates:
1. Docstrings and inline comments
2. Unit tests for the functions

**Week 4 Community Contribution by Mirack**

Skills demonstrated: Code generation, multi-model comparison, structured LLM output, streaming.

In [ ]:
import os
from openai import OpenAI

In [ ]:

client = OpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.environ.get("OPENAI_API_KEY")
)

MODEL = "gpt-4.1-nano"

## 1. Auto Docstring Generator
Paste any Python function, get back the same function with proper docstrings and comments.

In [ ]:
def add_docstrings(code, model=MODEL):
    """Take raw Python code and return it with docstrings and comments added."""
    prompt = f"""Add Google-style docstrings and brief inline comments to this Python code.
Return ONLY the improved code, no explanations, no markdown fences.

```python
{code}
```"""
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a senior Python developer. Add clear, concise docstrings and comments. Do not change the logic."},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )
    result = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="")
            result += content
    return result

In [ ]:
# Test with a raw function that has no docs
sample_code = """
def calculate_discount(price, discount_percent, min_price=0):
    if discount_percent < 0 or discount_percent > 100:
        raise ValueError("Invalid discount")
    discounted = price * (1 - discount_percent / 100)
    return max(discounted, min_price)

def merge_sorted(arr1, arr2):
    result = []
    i = j = 0
    while i < len(arr1) and j < len(arr2):
        if arr1[i] <= arr2[j]:
            result.append(arr1[i])
            i += 1
        else:
            result.append(arr2[j])
            j += 1
    result.extend(arr1[i:])
    result.extend(arr2[j:])
    return result
"""

documented_code = add_docstrings(sample_code)

## 2. Unit Test Generator
Given a Python function, auto-generate pytest unit tests.

In [ ]:
def generate_tests(code, model=MODEL):
    """Generate pytest unit tests for the given Python code."""
    prompt = f"""Write pytest unit tests for this Python code.
Cover: normal cases, edge cases, and error cases.
Return ONLY the test code, no explanations, no markdown fences.

```python
{code}
```"""
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a senior QA engineer. Write clean, thorough pytest tests."},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )
    result = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="")
            result += content
    return result

In [ ]:
print("Generated unit tests:\n")
tests = generate_tests(sample_code)

## 3. Multi-Model Comparison
Compare how different models handle the same code generation task.

In [ ]:
MODELS = {
    "GPT-4.1-nano": "gpt-4.1-nano",
    "Claude Sonnet": "anthropic/claude-3.5-sonnet"

}

small_func = """
def fibonacci(n):
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b
"""

for name, model_id in MODELS.items():
    print(f"\n{'='*40}")
    print(f"{name} - Docstrings:")
    print(f"{'='*40}")
   
    messages = []
    
    messages.append({"role": "system", "content": "You are a senior Python developer. Add docstrings."})
    messages.append({"role": "user", "content": f"Add Google-style docstrings to this code. Return ONLY code, no markdown:\n{small_func}"})
    
    stream = client.chat.completions.create(model=model_id, messages=messages, stream=True)
    for chunk in stream:
        if not chunk.choices:
            continue
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="")

## 4. Full Pipeline — Docstrings + Tests in One Go

In [ ]:
def full_code_review(code, model=MODEL):
    """Run the full pipeline: add docstrings then generate tests."""
    print("STEP 1: Adding docstrings...\n")
    documented = add_docstrings(code, model)
    print("\n\nSTEP 2: Generating unit tests...\n")
    tests = generate_tests(code, model)
    return documented, tests

user_code = """
def flatten(nested_list):
    flat = []
    for item in nested_list:
        if isinstance(item, list):
            flat.extend(flatten(item))
        else:
            flat.append(item)
    return flat
"""

doc_code, test_code = full_code_review(user_code)